# Hospital Patient Analytics & Healthcare Performance Optimization
## Comprehensive Exploratory Data Analysis (EDA) & Clinical Operational Insights

**Author:** Senior Healthcare Data Analyst & Data Engineer  
**Dataset:** MITRE Synthea Synthetic Patient Population (Strictly Synthetic, HIPAA De-identified)  
**Tech Stack:** Python 3.12, Pandas, NumPy, Matplotlib, Seaborn, SciPy, Scikit-learn

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

data_dir = Path('../data/processed')
print('Data Directory:', data_dir.resolve())

### 1. Load Processed & Enriched Datasets

In [ ]:
patients = pd.read_csv(data_dir / 'dim_patients_enriched.csv')
encounters = pd.read_csv(data_dir / 'fact_encounters_enriched.csv')
conditions = pd.read_csv(data_dir / 'dim_conditions_summary.csv')

encounters['START'] = pd.to_datetime(encounters['START'])
encounters['STOP'] = pd.to_datetime(encounters['STOP'])

print(f'Patients: {len(patients):,}')
print(f'Encounters: {len(encounters):,}')
print(f'Total Billed Charges: ${encounters["TOTAL_CLAIM_COST"].sum():,.2f}')

### 2. Patient Demographics & Age Bracket Distribution

In [ ]:
age_order = ['0-17 (Pediatric)', '18-34 (Young Adult)', '35-49 (Adult)', '50-64 (Middle-Aged)', '65-79 (Older Adult)', '80+ (Senior)']
demo = patients.groupby(['CURRENT_AGE_GROUP', 'GENDER']).size().unstack(fill_value=0).reindex(age_order)
demo.plot(kind='barh', stacked=False, color=['#0D9488', '#0284C7'])
plt.title('Patient Cohort Demographics by Age Group and Gender')
plt.xlabel('Patient Count')
plt.show()

### 3. Clinical Settings vs. Financial Billed Claims

In [ ]:
class_summary = encounters.groupby('ENCOUNTERCLASS').agg(
    Encounters=('Id', 'count'),
    Total_Claim_Cost=('TOTAL_CLAIM_COST', 'sum'),
    Mean_Duration_Hours=('DURATION_HOURS', 'mean')
).sort_values(by='Total_Claim_Cost', ascending=False)
class_summary

### 4. Hypothesis Testing: ANOVA across Care Settings

In [ ]:
groups = [g['TOTAL_CLAIM_COST'].values for _, g in encounters.groupby('ENCOUNTERCLASS')]
f_stat, p_val = stats.f_oneway(*groups)
print(f'One-way ANOVA F-statistic: {f_stat:.2f}, p-value: {p_val:.4e}')